# Setup


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



PyTorch: 2.14.1+cu130
GPU available: True
GPU: Tesla T4


In [2]:
!pip -q install -U transformers accelerate datasets
!pip -q install -U lm_eval

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import torch
import transformers
import datasets
import lm_eval

print("CUDA:", torch.cuda.is_available())
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("lm_eval:", lm_eval.__version__ if hasattr(lm_eval, "__version__") else "Imported successfully")

!lm-eval ls tasks | grep -E "gsm8k|ifeval"

CUDA: True
Transformers: 5.22.2
Datasets: 5.2.0
lm_eval: 0.4.13
|leaderboard_instruction_following                |lm_eval/tasks/leaderboard/ifeval/_leaderboard_instruction_following.yaml                                                                     |
|metabench_gsm8k_subset                          |
|gsm8k                                                                                 |lm_eval/tasks/gsm8k/gsm8k.yaml                                                                                                                                  |generate_until       |
|gsm8k_cot                                                                             |lm_eval/tasks/gsm8k/gsm8k-cot.yaml                                                                                                                              |generate_until       |
|gsm8k_cot_llama                                                                       |lm_eval/tasks/gsm8k/gsm8k-cot-llama.yaml                   

In [4]:
from transformers import AutoTokenizer

MODELS = {
    "Base": "allenai/OLMo-2-0425-1B",
    "SFT": "allenai/OLMo-2-0425-1B-SFT",
    "DPO": "allenai/OLMo-2-0425-1B-DPO",
    "Instruct": "allenai/OLMo-2-0425-1B-Instruct",
}

tokenizers = {}

for name, model_id in MODELS.items():
    print(f"\nLoading tokenizer: {name}")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizers[name] = tokenizer

    print("Tokenizer class:", tokenizer.__class__.__name__)
    print("Vocabulary size:", len(tokenizer))
    print("Has chat template:", tokenizer.chat_template is not None)

    if tokenizer.chat_template is not None:
        print("Template preview:", tokenizer.chat_template[:150])


Loading tokenizer: Base


config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.34k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Tokenizer class: TokenizersBackend
Vocabulary size: 100278
Has chat template: False

Loading tokenizer: SFT


... (11 outputs left out)

tokenizer_config.json:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/581 [00:00<?, ?B/s]

Tokenizer class: TokenizersBackend
Vocabulary size: 100278
Has chat template: True
Template preview: {{ bos_token }}{% for message in messages %}{% if message['role'] == 'system' %}{{ '<|system|>
' + message['content'] + '
' }}{% elif message['role'] 


In [5]:
messages = [
    {"role": "user", "content": "What is 12 multiplied by 8? Explain briefly."}
]

for name in ["SFT", "DPO", "Instruct"]:
    tokenizer = tokenizers[name]

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print(formatted)

print("\n==================== BASE ====================")
print(messages[0]["content"])


==================== SFT ====================
<|endoftext|><|user|>
What is 12 multiplied by 8? Explain briefly.
<|assistant|>


==================== DPO ====================
<|endoftext|><|user|>
What is 12 multiplied by 8? Explain briefly.
<|assistant|>


==================== Instruct ====================
<|endoftext|><|user|>
What is 12 multiplied by 8? Explain briefly.
<|assistant|>


==================== BASE ====================
What is 12 multiplied by 8? Explain briefly.


# Evaluation

In [6]:
!lm-eval run \
    --model hf \
    --model_args pretrained=allenai/OLMo-2-0425-1B-SFT,dtype=float16 \
    --tasks gsm8k \
    --device cuda \
    --batch_size 1 \
    --num_fewshot 0 \
    --limit 1 \
    --apply_chat_template \
    --gen_kwargs do_sample=False \
    --log_samples \
    --output_path /content/olmo_smoke_test

2026-11-11:04:53:42 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-11:04:53:42 INFO     [config.evaluate_config:357] Using default fewshot_as_multiturn=True.
2026-11-11:04:53:53 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-11:04:53:56 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-11:04:53:56 WARNING  [evaluator:226] generation_kwargs: {'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-11:04:53:56 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B-SFT', 'dtype': 'float16'}
2026-11-11:04:53:59 INFO     [models.huggingface:289] Using device 'cuda'
2026-11-11:04:54:00 INFO     [models.huggingface:582] Model parallel was 

In [7]:
import glob
import json

files = glob.glob(
    "/content/olmo_smoke_test/allenai__OLMo-2-0425-1B-SFT/**/*.jsonl",
    recursive=True
)

print("Sample files:", files)

for file in files:
    with open(file) as f:
        for line in f:
            sample = json.loads(line)
            print(json.dumps(sample, indent=2)[:4000])
            break

Sample files: ['/content/olmo_smoke_test/allenai__OLMo-2-0425-1B-SFT/samples_gsm8k_2026-11-11T04-55-08.646292.jsonl']
{
  "doc_id": 0,
  "doc": {
    "question": "Janet\u2019s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
    "answer": "Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer\u2019s market.\n#### 18"
  },
  "target": "Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer\u2019s market.\n#### 18",
  "arguments": {
    "gen_args_0": {
      "arg_0": "<|endoftext|><|user|>\nQuestion: Janet\u2019s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the r

In [8]:
# SFT on 200
!lm-eval run \
    --model hf \
    --model_args pretrained=allenai/OLMo-2-0425-1B-SFT,dtype=float16 \
    --tasks gsm8k \
    --device cuda \
    --batch_size 1 \
    --num_fewshot 0 \
    --limit 200 \
    --apply_chat_template \
    --gen_kwargs do_sample=False \
    --log_samples \
    --output_path /content/olmo_results/SFT_gsm8k

2026-11-11:05:00:44 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-11:05:00:44 INFO     [config.evaluate_config:357] Using default fewshot_as_multiturn=True.
2026-11-11:05:00:56 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-11:05:00:58 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-11:05:00:58 WARNING  [evaluator:226] generation_kwargs: {'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-11:05:00:58 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B-SFT', 'dtype': 'float16'}
2026-11-11:05:01:01 INFO     [models.huggingface:289] Using device 'cuda'
2026-11-11:05:01:02 INFO     [models.huggingface:582] Model parallel was 

In [9]:
# base model
!lm-eval run \
    --model hf \
    --model_args pretrained=allenai/OLMo-2-0425-1B,dtype=float16 \
    --tasks gsm8k \
    --device cuda \
    --batch_size 1 \
    --num_fewshot 0 \
    --limit 200 \
    --gen_kwargs do_sample=False \
    --log_samples \
    --output_path /content/olmo_results/Base_gsm8k

2026-11-11:05:11:10 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-11:05:11:26 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-11:05:11:29 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-11:05:11:29 WARNING  [evaluator:226] generation_kwargs: {'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-11:05:11:29 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B', 'dtype': 'float16'}
2026-11-11:05:11:33 INFO     [models.huggingface:289] Using device 'cuda'
2026-11-11:05:11:33 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda'}
model.safetensors.index.js

In [10]:
# DPO model
!lm-eval run \
    --model hf \
    --model_args pretrained=allenai/OLMo-2-0425-1B-DPO,dtype=float16 \
    --tasks gsm8k \
    --device cuda \
    --batch_size 1 \
    --num_fewshot 0 \
    --limit 200 \
    --apply_chat_template \
    --gen_kwargs do_sample=False \
    --log_samples \
    --output_path /content/olmo_results/DPO_gsm8k

2026-11-11:05:25:21 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-11:05:25:21 INFO     [config.evaluate_config:357] Using default fewshot_as_multiturn=True.
2026-11-11:05:25:46 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-11:05:25:52 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-11:05:25:52 WARNING  [evaluator:226] generation_kwargs: {'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-11:05:25:52 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B-DPO', 'dtype': 'float16'}
2026-11-11:05:25:58 INFO     [models.huggingface:289] Using device 'cuda'
2026-11-11:05:25:59 INFO     [models.huggingface:582] Model parallel was 

In [11]:
# instruct model
!lm-eval run \
    --model hf \
    --model_args pretrained=allenai/OLMo-2-0425-1B-Instruct,dtype=float16 \
    --tasks gsm8k \
    --device cuda \
    --batch_size 1 \
    --num_fewshot 0 \
    --limit 200 \
    --apply_chat_template \
    --gen_kwargs do_sample=False \
    --log_samples \
    --output_path /content/olmo_results/Instruct_gsm8k

2026-11-11:05:40:21 WARNING  [config.evaluate_config:337] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-11-11:05:40:21 INFO     [config.evaluate_config:357] Using default fewshot_as_multiturn=True.
2026-11-11:05:40:45 INFO     [_cli.run:392] Selected Tasks: ['gsm8k']
2026-11-11:05:40:52 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-11-11:05:40:52 WARNING  [evaluator:226] generation_kwargs: {'do_sample': False} specified through cli, these settings will update set parameters in yaml tasks. Ensure 'do_sample=True' for non-greedy decoding!
2026-11-11:05:40:52 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'allenai/OLMo-2-0425-1B-Instruct', 'dtype': 'float16'}
2026-11-11:05:40:59 INFO     [models.huggingface:289] Using device 'cuda'
2026-11-11:05:41:00 INFO     [models.huggingface:582] Model parallel